![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 3. Parte B. Preguntas y respuestas con BERT

**Asignatura:** Modelos de Lenguaje
**Bloque:** 3. Atención y Transformers

---

## Introducción

El etiquetador de la Parte A le agrega a BETO una cabeza que asigna una etiqueta a cada pieza. Este laboratorio le agrega otra cabeza, que marca en un párrafo dónde empieza y dónde termina la respuesta a una pregunta. El laboratorio llama **extractor** a BETO con esa cabeza.

La tarea que resuelve este laboratorio se llama **preguntas y respuestas extractivas**, en inglés *extractive question answering*. Cada ejemplo es una pregunta y un párrafo que contiene la respuesta, llamado **contexto**. La respuesta es un tramo del contexto. El modelo señala ese tramo sin generar texto. Por eso la tarea se llama extractiva.

Los ejemplos vienen de SQuAD, un corpus en inglés de preguntas sobre artículos de Wikipedia, con la respuesta de cada pregunta marcada en el párrafo. El laboratorio usa dos traducciones de SQuAD al español. SQuAD-es es una traducción automática del corpus entero. XQuAD es la traducción de una parte de SQuAD, hecha por traductores profesionales. Los ejemplos de entrenamiento salen de SQuAD-es. Los de evaluación salen de los dos corpus, en dos particiones separadas.

Respecto de la Parte A, el laboratorio cambia en cuatro aspectos.

- **La entrada.** Cada entrada reúne dos textos, la pregunta y el contexto, separados por `[SEP]`, como los pares de oraciones de la sección 2.2 de la Clase 2.
- **La salida.** La entrada del modelo es una sucesión de piezas. La **posición** de una pieza es su índice en esa sucesión, contado desde 0. Las posiciones cuentan piezas, no caracteres. La cabeza produce dos puntajes para cada posición. El puntaje de inicio indica qué tan probable es que la respuesta empiece en la pieza de esa posición. El puntaje de fin indica qué tan probable es que la respuesta termine en la pieza de esa posición. El puntaje de un tramo es la suma del puntaje de inicio de la posición de su primera pieza y el puntaje de fin de la posición de su última pieza. La respuesta predicha es el tramo de mayor puntaje cuyo fin no queda antes de su inicio.
- **El largo.** Algunos contextos superan el largo máximo de la entrada. Esos contextos se dividen en ventanas. Cada ventana repite la pregunta.
- **La métrica.** La métrica compara el texto de la respuesta predicha con las respuestas de referencia, palabra por palabra.

Al completar este laboratorio vas a poder:

- Dividir un contexto largo en ventanas y ubicar la respuesta de referencia en las piezas de cada ventana.
- Escribir sobre el encoder de BETO una cabeza que produce dos puntajes por posición, junto con su pérdida.
- Recuperar el texto de la respuesta a partir de los puntajes de todas las ventanas de una pregunta.
- Explicar qué información necesita el vector de cada posición para que una capa lineal ubique la respuesta.
- Revisar las respuestas de referencia de un corpus antes de reportar una cifra medida contra ellas.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **El laboratorio necesita GPU.** En Colab, *Entorno de ejecución > Cambiar tipo de entorno > T4 GPU*. El laboratorio afina tres modelos de 110 millones de parámetros, una época cada uno. En uno de los tres solo se entrena la cabeza. Con una T4, el total se estima del orden de cinco minutos. Antes de empezar, verificá que la primera celda de preparación imprima `cuda` como dispositivo.
- **La primera celda de preparación pide acceso a Google Drive.** La celda usa en Drive la carpeta `modelos-lenguaje-u3`, la misma de la Parte A, donde se guarda el resultado de cada afinado. Cada resultado incluye la configuración, la curva de la pérdida y los puntajes que el modelo asigna a cada posición de validación, y no incluye los pesos del modelo, que ocupan unos 440 MB. Si la sesión se interrumpe, al volver a ejecutar las celdas los afinados terminados se cargan de esa carpeta y no se repiten. Si no autorizás el acceso, la carpeta se crea en el disco de la sesión y se pierde al cerrarla.
- Este laboratorio va después de la Clase 2 de la Unidad 3 y de la Parte A. Da por vistos los pares de oraciones y los `token_type_ids` (secciones 1.1 y 2.2), el argumento `return_offsets_mapping` del tokenizador (sección 2.3), `ClasificadorBERT` y la receta de afinado (sección 3.1), y el congelado del encoder (sección 3.2). La tarea de preguntas y respuestas, las ventanas y la métrica no están en la clase. El laboratorio las explica antes de pedirlas.
- Usá las semillas que pide cada enunciado.
- Las cifras de un afinado varían entre máquinas y entre corridas, por los motivos que explica la sección «Sobre los números» de la Clase 2. Lo que se corrige es la dirección de cada resultado y el argumento con el que lo explicás.

## IMPORTANTE. Qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Las demás celdas, es decir, los enunciados, las explicaciones, los ejemplos provistos y el encabezado, **no se tocan**. La única excepción es el interruptor del bloque opcional del final, que el propio bloque explica.

La corrección se hace celda por celda. Cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo por separado, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las dos celdas que siguen vienen resueltas. No hay nada que completar en ellas, pero hay que ejecutarlas en orden antes de empezar. Cada una lleva antes una presentación con los nombres que deja definidos. Los ejercicios usan esos nombres.

La primera celda instala las bibliotecas que falten, las importa y prepara la carpeta de resultados. La segunda descarga los dos corpus.

El laboratorio tiene cuatro celdas de preparación más, ubicadas junto a los ejercicios que las necesitan. La primera, antes del Ejercicio 1, divide los contextos en ventanas. La segunda, antes del Ejercicio 2, define el bucle de afinado. La tercera, después del Ejercicio 3, define la métrica, que compara el texto de la respuesta predicha con las respuestas de referencia. La cuarta afina los dos extractores completos, porque necesita la clase y las funciones que escribís en los ejercicios 2 y 3. Al final hay un bloque opcional, que no se corrige.

| Nombre | Qué es |
|---|---|
| `DISPOSITIVO` | `"cuda"` si hay GPU; en otro caso, `"mps"` o `"cpu"` |
| `EN_GPU` | `True` si el dispositivo es CUDA. Activa la precisión mixta |
| `BETO` | el nombre en el Hub de BETO, el mismo de la Clase 2 |
| `CARPETA` | la carpeta donde se guardan los resultados de los afinados |
| `descargar` | descarga un archivo del laboratorio, salvo que ya exista en el disco de la sesión |

In [ ]:
# ─── Preparación. Bibliotecas, módulo de la materia y carpeta de resultados ─
import os, re, sys, math, time, json, hashlib, importlib, subprocess, unicodedata, urllib.request, warnings
from collections import Counter
from importlib.metadata import version, PackageNotFoundError


def instalar(*argumentos):
    """Instala con el pip del intérprete del notebook. Si pip falla, corta y muestra su error."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *argumentos],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(argumentos)} falló:\n{r.stderr[-3000:]}")
    importlib.invalidate_caches()                     # para que el import vea el paquete recién instalado


try:                                                  # transformers 5 o posterior, como la Clase 2
    mayor = int(version("transformers").split(".")[0])
except PackageNotFoundError:
    mayor = 0
if mayor < 5:
    instalar("-U", "transformers>=5")

os.environ.setdefault("HF_HUB_VERBOSITY", "error")
warnings.filterwarnings("ignore", message="Detected call of `lr_scheduler.step")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, BertModel, BertForQuestionAnswering

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)

# El repositorio de la materia da una URL estable. El Drive de la cátedra queda de
# respaldo para el módulo, porque Drive corta las descargas públicas por cuota.
RELEASES = "https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
HUB = "https://huggingface.co/datasets"
FUENTES = {                                     # archivo: (URL, ID de Drive)
    "cache_modelos.py":            (f"{RELEASES}/utils-v1/cache_modelos.py",
                                    "1k5wxbSMS-_3dC8FN_F-2VN6dp8tGFCpz"),
    "squad_es_train.parquet":      (f"{HUB}/ccasimiro/squad_es/resolve/refs%2Fconvert%2Fparquet/"
                                    "v1.1.0/train/0000.parquet", None),
    "squad_es_validation.parquet": (f"{HUB}/ccasimiro/squad_es/resolve/refs%2Fconvert%2Fparquet/"
                                    "v1.1.0/validation/0000.parquet", None),
    "xquad_es.parquet":            (f"{HUB}/google/xquad/resolve/main/"
                                    "xquad.es/validation-00000-of-00001.parquet", None),
}


def descargar(nombre):
    """Descarga un archivo de FUENTES, salvo que ya exista. Si la URL no responde, usa Drive."""
    if os.path.exists(nombre):
        return
    url, id_drive = FUENTES[nombre]
    try:
        urllib.request.urlretrieve(url, nombre)
    except Exception as falla:
        if id_drive is None:
            raise
        print(f"el repositorio de la materia no respondió ({falla}); se usa el Drive.")
        try:
            import gdown
        except ImportError:
            instalar("gdown")
            import gdown
        gdown.download(id=id_drive, output=nombre, quiet=False)


descargar("cache_modelos.py")
from cache_modelos import preparar_carpeta

if torch.cuda.is_available():
    DISPOSITIVO = "cuda"
elif torch.backends.mps.is_available():
    DISPOSITIVO = "mps"
else:
    DISPOSITIVO = "cpu"
EN_GPU = DISPOSITIVO == "cuda"                      # la precisión mixta se usa solo en CUDA

BETO = "dccuchile/bert-base-spanish-wwm-cased"      # el BERT en español
CARPETA = preparar_carpeta("modelos-lenguaje-u3")
torch.manual_seed(0)
if DISPOSITIVO == "cpu":
    print("Sin GPU. Cambiá el tipo de entorno a T4 antes de seguir.")

pd.Series({"transformers": transformers.__version__, "torch": torch.__version__,
           "dispositivo": DISPOSITIVO}).to_frame("versión")

### Los corpus SQuAD-es y XQuAD

Cada ejemplo de los dos corpus tiene una pregunta, un contexto y una o más respuestas de referencia. Cada respuesta de referencia es un tramo del contexto. El corpus indica además el índice del carácter del contexto en el que empieza ese tramo.

SQuAD-es tiene dos particiones, una de entrenamiento con 87.595 ejemplos y una de validación con 10.570. En la partición de entrenamiento, cada pregunta tiene una sola respuesta de referencia. En la de validación, cada pregunta tiene varias, porque en el SQuAD original varias personas marcaron la respuesta de cada pregunta. Los ejemplos guardan el carácter de inicio solo de la primera respuesta de referencia, que en la partición de entrenamiento es la única. El afinado necesita ese carácter para ubicar la respuesta en el contexto. La evaluación no lo necesita, porque compara solo los textos de las respuestas.

El laboratorio no usa las particiones enteras, porque afinar con todos los ejemplos llevaría demasiado tiempo. Usa 10.000 ejemplos de entrenamiento y 2.000 de validación, elegidos al azar. La elección al azar importa porque el corpus guarda juntas las preguntas de un mismo artículo de Wikipedia. Los primeros 10.000 ejemplos cubrirían pocos artículos.

XQuAD tiene 1.190 ejemplos, con una respuesta de referencia cada uno. Sus ejemplos son preguntas de la validación del SQuAD original, traducidas al español por traductores profesionales junto con su contexto y su respuesta.

El laboratorio usa los corpus de tres maneras.

- **Los 10.000 ejemplos de entrenamiento de SQuAD-es** sirven para afinar los extractores. Son los únicos ejemplos con los que el modelo aprende.
- **Los 2.000 ejemplos de validación de SQuAD-es** sirven para medir los extractores. Ninguna decisión del afinado se toma con ellos, así que cumplen el papel de una partición de prueba.
- **Los 1.190 ejemplos de XQuAD** sirven también para medir los extractores, como una segunda partición de prueba. Sus preguntas tienen el mismo origen que las de la validación de SQuAD-es, con otra traducción.

El Ejercicio 5 compara las respuestas de referencia de las dos particiones de prueba.

La celda descarga los tres archivos del Hub de Hugging Face en formato *parquet*, que ocupan 17 MB. Después imprime la cantidad de ejemplos de cada partición y un ejemplo de XQuAD.

| Nombre | Qué es |
|---|---|
| `ent`, `val`, `xquad` | las tres particiones, como listas de diccionarios, con un diccionario por ejemplo. Cada diccionario tiene cuatro claves. `"pregunta"` y `"contexto"` son textos. `"respuestas"` es la lista de los textos de las respuestas de referencia. `"car_inicio"` es el índice del carácter del contexto donde empieza la primera respuesta de referencia, contado desde 0 |
| `particiones` | un diccionario con las tres particiones, con las claves `"ent"`, `"val"` y `"xquad"` |
| `ent_completo` | los 87.595 ejemplos de entrenamiento, que usa solo el bloque opcional del final |
| `EJEMPLO` | el índice en `xquad` del ejemplo que imprime la celda |

In [ ]:
# ─── Preparación. SQuAD-es y XQuAD ─────────────────────────────────────────
def leer(archivo):
    """Un diccionario por ejemplo: pregunta, contexto, respuestas e índice del carácter de inicio de la primera."""
    descargar(archivo)
    tabla = pd.read_parquet(archivo)
    return [{"pregunta": f.question.strip(), "contexto": f.context,
             "respuestas": [str(r) for r in f.answers["text"]],
             "car_inicio": int(f.answers["answer_start"][0])} for f in tabla.itertuples()]


ent_completo = leer("squad_es_train.parquet")
val_completo = leer("squad_es_validation.parquet")
# SQuAD viene agrupado por artículo: las muestras son prefijos de una permutación al azar.
ent = [ent_completo[i] for i in np.random.default_rng(0).permutation(len(ent_completo))[:10_000]]
val = [val_completo[i] for i in np.random.default_rng(0).permutation(len(val_completo))[:2_000]]
xquad = leer("xquad_es.parquet")
particiones = {"ent": ent, "val": val, "xquad": xquad}

display(pd.DataFrame({clave: {"ejemplos": len(ej),
                              "contextos distintos": len({e["contexto"] for e in ej}),
                              "caracteres por contexto, media": round(np.mean([len(e["contexto"]) for e in ej]))}
                      for clave, ej in particiones.items()}))

EJEMPLO = next(i for i, e in enumerate(xquad) if len(e["contexto"]) < 600)
print("pregunta:  ", xquad[EJEMPLO]["pregunta"])
print("respuestas:", xquad[EJEMPLO]["respuestas"])
print("contexto:  ", xquad[EJEMPLO]["contexto"])

---
### Preparación. Las ventanas

La entrada del extractor es la pregunta seguida del contexto, con la forma `[CLS] pregunta [SEP] contexto [SEP]`.

Junto con el índice de cada pieza, el modelo recibe el **segmento** de cada posición, un 0 o un 1 que indica a cuál de los dos textos pertenece la posición. La sección 1.1 de la Clase 2 presentó los segmentos, que el modelo recibe como `token_type_ids`. En el extractor, el primer texto es la pregunta y el segundo es el contexto. `[CLS]` y el primer `[SEP]` tienen el segmento de la pregunta. El último `[SEP]` tiene el segmento del contexto.

```
pieza     [CLS]  pregunta  [SEP]  contexto  [SEP]
segmento    0       0        0       1        1
```

El segmento se pasa como un índice, no como un vector. BETO tiene una tabla con un *embedding* aprendido por segmento, como mostró la sección 1.4 de la Clase 2. El *embedding* de segmento que elige el índice de cada posición se suma a los de la pieza y de la posición. La función `armar_ventanas` de esta celda guarda los segmentos de cada ventana en la clave `"segmentos"`.

La sección 1.4 de la Clase 2 mostró que BETO acepta entradas de hasta 512 posiciones. Este laboratorio limita la entrada a 384 posiciones, el valor de `LARGO_MAX`, porque el costo de la autoatención crece con el cuadrado del largo de la entrada, como explicó la sección 1.1 de la Clase 1.

Un contexto que no entra junto con la pregunta en `LARGO_MAX` posiciones se divide en **ventanas**. Cada ventana lleva la pregunta completa y un tramo del contexto. Dos ventanas consecutivas de una misma pregunta comparten 128 piezas del contexto, el valor de `SOLAPE`.

La función `armar_ventanas` construye las ventanas en tres pasos. Primero tokeniza la pregunta y el contexto por separado, con el argumento `add_special_tokens=False`, que omite `[CLS]` y `[SEP]`. Después recorre la lista de piezas del contexto en tramos de igual largo, el máximo que entra junto con la pregunta. Por último, construye cada ventana con la pregunta, el tramo y los tokens especiales.

Cada ejemplo del corpus indica el índice del carácter del contexto en el que empieza la primera respuesta de referencia, con la clave `"car_inicio"`. El fin de la respuesta sale del largo de su texto. El carácter que sigue al último tiene el índice `"car_inicio"` más ese largo. El extractor, en cambio, predice dos posiciones de la ventana, la de la primera pieza de la respuesta y la de la última. Para pasar la respuesta de caracteres a posiciones, hace falta saber en qué índice de carácter del contexto empieza y termina cada pieza.

Un **tramo** es un rango de índices de carácter del contexto. Un tramo se escribe como un par `(a, b)`. El índice `a` es el del primer carácter. El índice `b` es el del carácter que sigue al último, como en un *slice* de Python, así que `contexto[a:b]` es el texto del tramo. La respuesta de referencia es un tramo. Cada pieza del contexto ocupa también un tramo. La función `armar_ventanas` guarda en la clave `"offsets"` el tramo de cada pieza de la ventana. Los obtiene con el argumento `return_offsets_mapping` de la sección 2.3 de la Clase 2. Cada elemento de `"offsets"` es la lista de los tramos de una ventana, con uno por pieza y en el orden de la ventana. Las piezas `[CLS]`, las de la pregunta y los dos `[SEP]` no son del contexto y llevan `None` en lugar de un tramo.

La tabla muestra la ventana de la pregunta «¿Dónde vive Camila?» con el contexto «Camila vive en Godoy Cruz y trabaja en Mendoza.», tokenizados con BETO.

| Posición | Pieza | Tramo | `contexto[a:b]` |
|---|---|---|---|
| 0 | `[CLS]` | `None` | |
| 1 a 6 | las seis piezas de la pregunta | `None` | |
| 7 | `[SEP]` | `None` | |
| 8 | `Cami` | `(0, 4)` | `Cami` |
| 9 | `##la` | `(4, 6)` | `la` |
| 10 | `vive` | `(7, 11)` | `vive` |
| 11 | `en` | `(12, 14)` | `en` |
| 12 | `God` | `(15, 18)` | `God` |
| 13 | `##oy` | `(18, 20)` | `oy` |
| 14 | `Cruz` | `(21, 25)` | `Cruz` |
| 15 | `y` | `(26, 27)` | `y` |
| 16 | `trabaja` | `(28, 35)` | `trabaja` |
| 17 | `en` | `(36, 38)` | `en` |
| 18 | `Mendoza` | `(39, 46)` | `Mendoza` |
| 19 | `.` | `(46, 47)` | `.` |
| 20 | `[SEP]` | `None` | |

El tramo de una pieza que continúa una palabra no incluye los dos `#`, que no están en el contexto. Los espacios no pertenecen a ninguna pieza. Por eso el tramo de `vive` empieza en el índice 7 y no en el 6, que es el espacio. La respuesta «Godoy Cruz» es el tramo `(15, 25)`. El índice 15 es su `"car_inicio"`. El índice 25 es `"car_inicio"` más el largo de «Godoy Cruz», que tiene 10 caracteres. El Ejercicio 1 usa los tramos de las piezas para calcular las posiciones en las que empieza y termina la respuesta. En esta ventana, la primera pieza de «Godoy Cruz» está en la posición 12 y la última en la posición 14.

La celda aplica `armar_ventanas` a las tres particiones e imprime cuántas ventanas tiene cada una y en cuántos ejemplos el contexto se divide en más de una ventana. Después muestra las ventanas de un ejemplo de entrenamiento cuyo contexto se divide en dos ventanas, con el primer y el último carácter del tramo del contexto de cada una.

| Nombre | Qué es |
|---|---|
| `tok_beto` | el tokenizador de BETO |
| `PAD` | el índice de relleno de BETO, que vale 1 |
| `LARGO_MAX`, `SOLAPE` | 384 y 128 |
| `armar_ventanas(ejemplos, tok, ubicar=None)` | la función que divide los ejemplos en ventanas. El argumento `ejemplos` tiene el formato de `val`. La función devuelve un diccionario, que se describe debajo de la tabla |
| `EJEMPLO_LARGO` | el índice en `ent` del ejemplo cuyas ventanas muestra la celda |

El diccionario que devuelve `armar_ventanas` tiene cuatro claves. El valor de cada clave es una lista con un elemento por ventana. En `"ids"`, `"segmentos"` y `"offsets"`, cada elemento es a su vez una lista, con un elemento por pieza de la ventana.

- `"ids"` guarda el índice de cada pieza en el vocabulario.
- `"segmentos"` guarda el segmento de cada pieza, 0 o 1.
- `"offsets"` guarda el tramo de cada pieza, o `None` si la pieza no es del contexto.
- `"ejemplo"` guarda, para cada ventana, el índice del diccionario de `ejemplos` del que salió.

Por ejemplo, si `v = armar_ventanas(val, tok_beto)`, `v["ids"][k][p]` es el índice de la pieza en la posición `p` de la ventana `k`. El valor `v["ejemplo"][k]` es el índice en `val` del diccionario del que salió la ventana `k`.

In [ ]:
# ─── Preparación. La pregunta y el contexto, divididos en ventanas ──────────
tok_beto = AutoTokenizer.from_pretrained(BETO)
PAD, LARGO_MAX, SOLAPE = tok_beto.pad_token_id, 384, 128


def armar_ventanas(ejemplos, tok, ubicar=None):
    """[CLS] pregunta [SEP] tramo del contexto [SEP], con SOLAPE piezas compartidas entre tramos.

    Si recibe ubicar(tramos, car_inicio, car_fin), agrega las claves "inicio" y "fin".
    """
    preguntas = tok([e["pregunta"] for e in ejemplos], add_special_tokens=False)["input_ids"]
    contextos = tok([e["contexto"] for e in ejemplos], add_special_tokens=False,
                    return_offsets_mapping=True)
    ventanas = {"ids": [], "segmentos": [], "offsets": [], "ejemplo": []}
    if ubicar is not None:
        ventanas.update(inicio=[], fin=[])
    for i, e in enumerate(ejemplos):
        pregunta = preguntas[i]
        piezas, tramos = contextos["input_ids"][i], [tuple(t) for t in contextos["offset_mapping"][i]]
        n = LARGO_MAX - len(pregunta) - 3                 # piezas del contexto por ventana
        comienzo = 0
        while True:
            ids, offsets = piezas[comienzo:comienzo + n], tramos[comienzo:comienzo + n]
            ventanas["ids"].append([tok.cls_token_id] + pregunta + [tok.sep_token_id] + ids + [tok.sep_token_id])
            ventanas["segmentos"].append([0] * (len(pregunta) + 2) + [1] * (len(ids) + 1))
            ventanas["offsets"].append([None] * (len(pregunta) + 2) + offsets + [None])
            ventanas["ejemplo"].append(i)
            if ubicar is not None:
                car_inicio = e["car_inicio"]
                inicio, fin = ubicar(ventanas["offsets"][-1], car_inicio,
                                     car_inicio + len(e["respuestas"][0]))
                ventanas["inicio"].append(inicio)
                ventanas["fin"].append(fin)
            if comienzo + n >= len(piezas):              # el tramo llegó al final del contexto
                break
            comienzo += n - SOLAPE
    return ventanas


def tramo_del_contexto(offsets):
    """El primer carácter y el carácter siguiente al último del tramo del contexto de una ventana."""
    tramos = [t for t in offsets if t is not None]
    return tramos[0][0], tramos[-1][1]


sin_ubicar = {clave: armar_ventanas(ej, tok_beto) for clave, ej in particiones.items()}
display(pd.DataFrame({clave: {"ventanas": len(v["ids"]),
                              "preguntas con más de una ventana": sum(c > 1 for c in Counter(v["ejemplo"]).values())}
                      for clave, v in sin_ubicar.items()}))

# El primer ejemplo de entrenamiento con dos ventanas y la respuesta dentro de las dos.
v = sin_ubicar["ent"]
de_ejemplo = {}
for k, i in enumerate(v["ejemplo"]):
    de_ejemplo.setdefault(i, []).append(k)
EJEMPLO_LARGO = next(
    i for i, ks in de_ejemplo.items() if len(ks) == 2 and all(
        tramo_del_contexto(v["offsets"][k])[0] <= ent[i]["car_inicio"] and
        ent[i]["car_inicio"] + len(ent[i]["respuestas"][0]) <= tramo_del_contexto(v["offsets"][k])[1]
        for k in ks))
e = ent[EJEMPLO_LARGO]
print("pregunta: ", e["pregunta"])
print("respuesta:", e["respuestas"][0], f"(caracteres {e['car_inicio']} a "
      f"{e['car_inicio'] + len(e['respuestas'][0])}, de {len(e['contexto'])})")
pd.DataFrame([{"piezas": len(v["ids"][k]), "primer carácter del tramo": tramo_del_contexto(v["offsets"][k])[0],
               "carácter siguiente al último": tramo_del_contexto(v["offsets"][k])[1]}
              for k in de_ejemplo[EJEMPLO_LARGO]]).rename_axis("ventana")

### Ejercicio 1. De caracteres a posiciones

**Objetivo.** Calcular, en cada ventana, la posición de la pieza en la que empieza la respuesta de referencia y la posición de la pieza en la que termina.

La pérdida del extractor necesita dos posiciones por ventana, la de la primera pieza de la respuesta y la de la última. Esas posiciones se llaman `inicio` y `fin`. Cuando la respuesta no está entera en una ventana, la convención habitual es que `inicio` y `fin` valgan 0, que es la posición de `[CLS]`. El Ejercicio 2 explica cómo usa la pérdida esas dos posiciones.

**Enunciado:**

1. Escribí la función `ubicar(tramos, car_inicio, car_fin)`, que devuelve la tupla `(inicio, fin)` de una ventana, con la posición de la primera pieza de la respuesta y la posición de la última. La función recibe tres argumentos:
   - `tramos` es un elemento de la clave `"offsets"`, la lista de los tramos de las piezas de la ventana. Las posiciones del contexto son las que tienen un tramo distinto de `None`.
   - `car_inicio` y `car_fin` forman el tramo de la respuesta, `(car_inicio, car_fin)`.

   La función calcula `(inicio, fin)` en dos pasos. En los dos, `(a, b)` es `tramos[p]`, el tramo de la pieza en la posición `p` que se evalúa.
   - La respuesta está entera en la ventana si se cumplen dos condiciones. El tramo de la primera posición del contexto cumple `a <= car_inicio`. El tramo de la última posición del contexto cumple `b >= car_fin`. Si la respuesta no está entera, la función devuelve `(0, 0)`.
   - Si la respuesta está entera, `inicio` es la última posición del contexto cuyo tramo cumple `a <= car_inicio`. `fin` es la primera posición del contexto cuyo tramo cumple `b >= car_fin`.

2. Creá un diccionario llamado `ventanas_beto`, con las claves `"ent"`, `"val"` y `"xquad"`. El valor de cada clave es el resultado de `armar_ventanas` sobre la partición correspondiente de `particiones`, con `tok_beto` y con tu función `ubicar` como tercer argumento. Con `ubicar`, `armar_ventanas` agrega las claves `"inicio"` y `"fin"`, con el resultado de `ubicar` para cada ventana.

3. Verificá la función. Para una ventana con la respuesta entera, el **texto recuperado** es el texto del contexto desde el primer carácter de la pieza en la posición `inicio` hasta el último carácter de la pieza en la posición `fin`.
   - Mostrá, para las dos ventanas del ejemplo `ent[EJEMPLO_LARGO]`, una tabla con una fila por ventana y tres columnas, `inicio`, `fin` y el texto recuperado.
   - Contá cuántas ventanas de `ventanas_beto["ent"]` no contienen la respuesta entera. Para las demás, contá en cuántas el texto recuperado coincide exactamente con la primera respuesta de referencia de su ejemplo. Imprimí esos dos números junto con la cantidad total de ventanas de entrenamiento. Si `ubicar` está bien escrita, más del 90 % de las ventanas con la respuesta entera recuperan exactamente la respuesta de referencia.
   - Imprimí cinco pares de respuesta de referencia y texto recuperado que no coinciden.

> **Pista.** Las posiciones del contexto aparecen en la ventana en el mismo orden que en el texto, así que el primer carácter de sus piezas crece con la posición.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

En el ejemplo `ent[EJEMPLO_LARGO]`, la respuesta está entera en las dos ventanas, porque cae en el tramo de 128 piezas que las dos comparten. Explicá qué pasaría, en el entrenamiento y en la predicción, con una respuesta ubicada en la frontera entre dos ventanas si las ventanas no compartieran piezas, y qué respuestas quedan siempre enteras en alguna ventana con un solape de 128 piezas.

*(Escribí tu respuesta acá)*

---
### Preparación. El bucle de afinado

La celda que sigue define el bucle de afinado con la receta de la sección 3.1 de la Clase 2. El optimizador es AdamW, con un `lr` de `2e-5`. Durante el primer 10 % de los pasos, el `lr` sube desde cero hasta `2e-5`, que es el calentamiento. Después, el `lr` baja linealmente hasta cero. En la GPU, el bucle usa precisión mixta. Además, el bucle recorta la norma del gradiente en 1, como en la Clase 2 de la Unidad 2. Los lotes se construyen con ventanas de largo parecido, como en la Parte 3 de la Clase 2, para reducir el relleno. El afinado dura una época, como en la Clase 2, con lotes de 16 ventanas. Una época sobre las ventanas de los 10.000 ejemplos de entrenamiento son 635 pasos.

Los ejercicios afinan los modelos con la función `afinar`, que encadena el trabajo de los ejercicios 2 y 3. Primero, `afinar` fija la semilla y construye el modelo con la función `construir` que recibe. Después, `afinar` entrena el modelo sobre las ventanas de entrenamiento. En cada paso, el bucle arma un lote y calcula su pérdida con la función `perdida_de` que recibe `afinar`, que para tu extractor es la pérdida del Ejercicio 2. Al terminar el entrenamiento, `afinar` calcula los logits de inicio y de fin de cada ventana de validación y de XQuAD. Por último, `afinar` convierte esos logits en el texto de una respuesta por pregunta, con tu función `responder` del Ejercicio 3.

Cada lote es la tupla `(ids, mascara, segmentos, inicio, fin)` que devuelve `construir_lote`. Los cinco tensores tienen una fila por ventana del lote, en el mismo orden, así que la fila `k` de cada tensor corresponde a la misma ventana. `ids`, `mascara` y `segmentos` tienen forma `(lote, L)`, donde `L` es el largo de la ventana más larga del lote. Las ventanas más cortas se completan con relleno, en el que la máscara vale 0. `inicio` y `fin` tienen forma `(lote,)`, con las dos posiciones que calculó `ubicar` para cada ventana.

`afinar` guarda en `CARPETA` los logits de cada posición, la configuración, la curva de la pérdida y el tiempo de afinado. Los pesos no se guardan. Si los archivos ya existen y la configuración coincide, `afinar` los carga y no afina. La conversión de los logits en texto, en cambio, se repite en cada llamada. Por eso una corrección de `responder` no requiere volver a afinar.

La configuración incluye una **huella** del código del modelo y de la pérdida, y otra de las ventanas. Si corregís tu clase, tu pérdida o `ubicar` después de haber afinado un modelo, la huella cambia. Con la huella distinta, `afinar` vuelve a afinar el modelo, en lugar de cargar un resultado obtenido con el código anterior.

| Nombre | Qué es |
|---|---|
| `LOTE`, `EPOCAS`, `LR` | 16, 1 y `2e-5` |
| `N_MEJORES`, `LARGO_MAX_TRAMO` | 20 y 30, los dos topes que el Ejercicio 3 impone a la búsqueda del tramo predicho |
| `construir_lote(ventanas, indices)` | la función que arma el lote con las ventanas `indices` de una partición, como `ventanas_beto["val"]` |
| `afinar(nombre, construir, perdida_de, ventanas, semilla, lr=LR)` | la función que afina un modelo o carga su resultado guardado. `construir` es una función sin argumentos que devuelve el modelo sin afinar. `ventanas` es un diccionario como `ventanas_beto`. `afinar` devuelve un diccionario con las claves `"val"` y `"xquad"`, que contienen la respuesta predicha para cada ejemplo, en el orden de la partición, y con las claves `"historia"`, `"minutos"` y `"entrenables"` |

La celda define además cuatro funciones que ningún ejercicio llama. `afinar` usa por dentro `entrenar`, `logits_de` y `predecir_logits`. La preparación posterior al Ejercicio 3 usa `perdida_hf`, que es la pérdida de `BertForQuestionAnswering`.

In [ ]:
# ─── Preparación. Los lotes, el bucle de afinado y la predicción ────────────
LOTE, EPOCAS, LR = 16, 1, 2e-5
N_MEJORES, LARGO_MAX_TRAMO = 20, 30


def rellenar(filas, valor):
    """Lista de listas de enteros → tensor (lote, L), rellenado con `valor` hasta la fila más larga."""
    L = max(len(f) for f in filas)
    x = torch.full((len(filas), L), valor, dtype=torch.long)
    for i, f in enumerate(filas):
        x[i, :len(f)] = torch.tensor(f)
    return x.to(DISPOSITIVO)


def construir_lote(ventanas, indices):
    """(ids, máscara, segmentos, inicio, fin) de las ventanas `indices`."""
    ids = rellenar([ventanas["ids"][i] for i in indices], PAD)
    segmentos = rellenar([ventanas["segmentos"][i] for i in indices], 0)
    inicio = torch.tensor([ventanas["inicio"][i] for i in indices], device=DISPOSITIVO)
    fin = torch.tensor([ventanas["fin"][i] for i in indices], device=DISPOSITIVO)
    return ids, (ids != PAD).long(), segmentos, inicio, fin


def lotes_por_largo(ventanas, lote, semilla):
    """Índices de ventanas agrupados en lotes de largo parecido, con los lotes en orden al azar."""
    gen = np.random.default_rng(semilla)
    largos = np.array([len(x) for x in ventanas["ids"]]) + gen.random(len(ventanas["ids"]))
    orden = np.argsort(largos)                       # el sumando al azar desempata los largos iguales
    lotes = [orden[i:i + lote] for i in range(0, len(orden), lote)]
    return [lotes[k] for k in gen.permutation(len(lotes))]


def entrenar(modelo, ventanas, perdida_de, semilla, lr):
    """Afina los parámetros con requires_grad=True, con la receta de la Clase 2 (sección 3.1).

    ventanas: las de entrenamiento. perdida_de(modelo, lote) devuelve la pérdida del lote.
    Devuelve la historia de la pérdida, los segundos y la memoria pico de la GPU.
    """
    parametros = [p for p in modelo.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(parametros, lr=lr, weight_decay=0.01)
    pasos = EPOCAS * math.ceil(len(ventanas["ids"]) / LOTE)
    calentamiento = max(1, pasos // 10)
    programa = torch.optim.lr_scheduler.LambdaLR(   # multiplicador del lr: de 0 a 1, y de 1 a 0
        opt, lambda p: min((p + 1) / calentamiento, max(0.0, (pasos - p) / (pasos - calentamiento))))
    escalador = torch.amp.GradScaler(enabled=EN_GPU)
    if EN_GPU:
        torch.cuda.reset_peak_memory_stats()
    historia, paso, acumulada, t0 = [], 0, 0.0, time.time()
    modelo.train()
    for epoca in range(EPOCAS):
        for indices in lotes_por_largo(ventanas, LOTE, semilla + epoca):
            lote = construir_lote(ventanas, indices)
            with torch.autocast("cuda" if EN_GPU else "cpu", dtype=torch.float16, enabled=EN_GPU):
                perdida = perdida_de(modelo, lote)
            opt.zero_grad(set_to_none=True)
            escalador.scale(perdida).backward()
            escalador.unscale_(opt)
            nn.utils.clip_grad_norm_(parametros, 1.0)
            escalador.step(opt)
            escalador.update()
            programa.step()
            paso += 1
            acumulada += perdida.item()
            if paso % 100 == 0 or paso == pasos:
                n = paso - (historia[-1]["paso"] if historia else 0)
                historia.append({"paso": paso, "pérdida": round(acumulada / n, 4)})
                print(f"   paso {paso:5d}/{pasos}  pérdida {acumulada / n:.3f}", flush=True)
                acumulada = 0.0
    return {"historia": historia, "segundos": round(time.time() - t0, 1),
            "memoria_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if EN_GPU else None}


def logits_de(modelo, ids, mascara, segmentos):
    """(logits de inicio, logits de fin), cada uno (lote, L)."""
    if isinstance(modelo, BertForQuestionAnswering):
        salida = modelo(input_ids=ids, attention_mask=mascara, token_type_ids=segmentos)
        return salida.start_logits, salida.end_logits
    return modelo(ids, mascara, segmentos)


def perdida_hf(modelo, lote):
    """La pérdida de BertForQuestionAnswering, que la clase calcula al recibir las dos posiciones."""
    ids, mascara, segmentos, inicio, fin = lote
    return modelo(input_ids=ids, attention_mask=mascara, token_type_ids=segmentos,
                  start_positions=inicio, end_positions=fin).loss


@torch.no_grad()
def predecir_logits(modelo, ventanas, lote=64):
    """Un par (l_inicio, l_fin) de arreglos de NumPy por ventana, sin las posiciones de relleno."""
    modelo.eval()
    salida = [None] * len(ventanas["ids"])
    orden = np.argsort([len(x) for x in ventanas["ids"]])
    for i0 in range(0, len(orden), lote):
        indices = orden[i0:i0 + lote]
        ids = rellenar([ventanas["ids"][i] for i in indices], PAD)
        segmentos = rellenar([ventanas["segmentos"][i] for i in indices], 0)
        with torch.autocast("cuda" if EN_GPU else "cpu", dtype=torch.float16, enabled=EN_GPU):
            l_inicio, l_fin = logits_de(modelo, ids, (ids != PAD).long(), segmentos)
        l_inicio, l_fin = l_inicio.float().cpu().numpy(), l_fin.float().cpu().numpy()
        for fila, i in enumerate(indices):
            L = len(ventanas["ids"][i])
            salida[i] = (l_inicio[fila, :L], l_fin[fila, :L])
    return salida


# ─── La huella. Cambia si cambia el código o los datos, no los comentarios ──
def _bytecode(codigo):
    partes = [codigo.co_code, " ".join(codigo.co_names).encode()]
    for k in codigo.co_consts:
        partes.append(_bytecode(k) if hasattr(k, "co_code") else repr(k).encode())
    return b"".join(partes)


def huella(*funciones):
    """Resumen del bytecode de las funciones, de los métodos de las clases del notebook
    que nombran y de los textos globales que usan, como el nombre de un modelo."""
    vistas, partes, pendientes = set(), [], list(funciones)
    while pendientes:
        f = pendientes.pop(0)
        if id(f) in vistas:
            continue
        vistas.add(id(f))
        if isinstance(f, type):
            pendientes += [v for v in vars(f).values() if callable(v)]
            continue
        codigo = getattr(f, "__code__", None)
        if codigo is None:
            continue
        partes += [_bytecode(codigo), repr(getattr(f, "__defaults__", None)).encode()]
        for nombre in codigo.co_names:
            valor = f.__globals__.get(nombre)
            if isinstance(valor, str):
                partes.append(valor.encode())
            elif isinstance(valor, type) and valor.__module__ == "__main__":
                pendientes.append(valor)
    return hashlib.sha256(b"".join(partes)).hexdigest()[:16]


def huella_ventanas(ventanas):
    texto = json.dumps([ventanas["ent"]["ids"], ventanas["ent"]["inicio"], ventanas["ent"]["fin"],
                        ventanas["val"]["ids"], ventanas["xquad"]["ids"]])
    return hashlib.sha256(texto.encode()).hexdigest()[:16]


# ─── Afinar, o cargar el resultado guardado ────────────────────────────────
def afinar(nombre, construir, perdida_de, ventanas, semilla, lr=LR):
    """Afina el modelo que devuelve construir() y calcula los logits de validación y de XQuAD.

    Guarda en CARPETA un .json con la configuración, la historia y el tiempo, y un .npz
    con los logits por posición. No guarda los pesos. Si los dos archivos existen y la
    configuración coincide, los carga y no afina.
    Devuelve un diccionario con "val" y "xquad" (la respuesta predicha para cada ejemplo,
    con responder), "historia", "minutos" y "entrenables".
    """
    config = {"semilla": semilla, "lote": LOTE, "epocas": EPOCAS, "lr": lr,
              "largo_max": LARGO_MAX, "solape": SOLAPE, "ventanas_ent": len(ventanas["ent"]["ids"]),
              "codigo": huella(construir, perdida_de), "datos": huella_ventanas(ventanas)}
    ruta, ruta_logits = (os.path.join(CARPETA, f"{nombre}.{ext}") for ext in ("json", "npz"))
    r = None
    if os.path.exists(ruta) and os.path.exists(ruta_logits):
        with open(ruta, encoding="utf-8") as archivo:
            guardado = json.load(archivo)
        if guardado["config"] == config:
            r = guardado
            print(f"[{nombre}] cargado de {ruta} (afinado el {r['fecha']})")
        else:
            print(f"[{nombre}] cambió la configuración, el código o los datos: se vuelve a afinar")
    if r is None:
        torch.manual_seed(semilla)
        modelo = construir().to(DISPOSITIVO)
        print(f"[{nombre}] afinando {EPOCAS} época en {DISPOSITIVO}", flush=True)
        r = entrenar(modelo, ventanas["ent"], perdida_de, semilla, lr)
        r.update(config=config, fecha=time.strftime("%Y-%m-%d %H:%M"),
                 entrenables=sum(p.numel() for p in modelo.parameters() if p.requires_grad))
        arreglos = {}
        for clave in ("val", "xquad"):
            pares = predecir_logits(modelo, ventanas[clave])
            arreglos[f"{clave}_inicio"] = np.concatenate([a for a, _ in pares]).astype(np.float16)
            arreglos[f"{clave}_fin"] = np.concatenate([b for _, b in pares]).astype(np.float16)
        del modelo
        if EN_GPU:
            torch.cuda.empty_cache()
        with open(ruta, "w", encoding="utf-8") as archivo:
            json.dump(r, archivo)
        np.savez_compressed(ruta_logits, **arreglos)
        print(f"[{nombre}] {r['segundos'] / 60:.1f} minutos, guardado en {ruta}")
    arreglos = np.load(ruta_logits)
    for clave in ("val", "xquad"):
        cortes = np.cumsum([len(x) for x in ventanas[clave]["ids"]])[:-1]
        logits = list(zip(np.split(arreglos[f"{clave}_inicio"].astype(np.float32), cortes),
                          np.split(arreglos[f"{clave}_fin"].astype(np.float32), cortes)))
        r[clave] = responder(logits, ventanas[clave], particiones[clave])
    r["minutos"] = round(r["segundos"] / 60, 1)
    return r

### Ejercicio 2. Dos puntajes por posición

**Objetivo.** Escribir el extractor, que agrega una cabeza al encoder de BETO, y su pérdida.

`EtiquetadorBERT`, en la Parte A, aplica al vector de cada posición una capa lineal con una salida por etiqueta. El extractor aplica al vector de cada posición una capa lineal con dos salidas. La primera salida es el puntaje de que la respuesta empiece en esa posición. La segunda salida es el puntaje de que la respuesta termine en esa posición. Los dos puntajes son logits, porque la pérdida les aplica una softmax.

La pérdida de una ventana resuelve dos clasificaciones. La primera clasificación elige, entre las posiciones de la ventana, la posición `inicio`. La segunda clasificación elige la posición `fin`. Cada clasificación aporta una entropía cruzada. La pérdida de la ventana es el promedio de las dos.

**Enunciado:**

1. Escribí la clase `ExtractorBERT`, que hereda de `nn.Module`. A diferencia de `ClasificadorBERT` y de `EtiquetadorBERT`, la clase no aplica *dropout* antes de la cabeza. Así tiene la misma arquitectura que `BertForQuestionAnswering`, la clase de Hugging Face con la que se compara después del Ejercicio 3.
   - Su método `__init__(self, nombre=BETO)` crea dos atributos:
     - `encoder`, el `BertModel` preentrenado que indica `nombre`, sin la capa *pooler*, como en `ClasificadorBERT`;
     - `cabeza`, una `nn.Linear` con `encoder.config.hidden_size` entradas y 2 salidas.
   - Su método `forward(self, ids, mascara, segmentos)` pasa los segmentos al encoder como `token_type_ids` y aplica la cabeza al vector de salida de cada posición. El método devuelve una tupla de dos tensores de forma `(lote, L)`, los logits de inicio y los logits de fin.

   > **Pista.** La cabeza devuelve un tensor de forma `(lote, L, 2)`, con los dos logits de cada posición en la última dimensión. El método `unbind(-1)` separa ese tensor en los logits de inicio y los logits de fin, dos tensores de forma `(lote, L)`.

2. Escribí la función `perdida_extractor(modelo, lote)`. La función recibe un `ExtractorBERT` y una tupla `(ids, mascara, segmentos, inicio, fin)` como la que devuelve `construir_lote`. La función pasa `ids`, `mascara` y `segmentos` al modelo y devuelve el promedio de dos entropías cruzadas calculadas con `F.cross_entropy`, la de los logits de inicio contra `inicio` y la de los logits de fin contra `fin`. En cada una de las dos clasificaciones, las clases son las `L` posiciones de la ventana. Por eso los logits, de forma `(lote, L)`, y las posiciones correctas, de forma `(lote,)`, ya tienen las formas `(N, C)` y `(N,)` que espera `F.cross_entropy`. A diferencia de la pérdida del etiquetador de la Parte A, esta función no une ninguna dimensión.
3. Verificá la clase y la pérdida.
   - Fijá la semilla de PyTorch en 0, creá un `ExtractorBERT` en una variable llamada `demo` y llevalo a `DISPOSITIVO`.
   - Construí con `construir_lote` un lote con las ventanas 0, 1, 2 y 3 de `ventanas_beto["val"]`.
   - Imprimí la forma de los dos tensores de logits, el valor de la pérdida y $\ln L$, con $L$ igual a la segunda dimensión de los logits, el largo de la ventana más larga del lote. $\ln L$ es la entropía cruzada de una distribución uniforme sobre las $L$ posiciones de una ventana. Con la cabeza sin afinar, la pérdida tiene que estar cerca de $\ln L$, a menos de medio punto.
   - Al terminar, borrá el modelo con `del demo`, para liberar la memoria de la GPU antes de los afinados.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

En la Parte A, el etiquetador clasifica cada posición por separado entre nueve etiquetas. El extractor podría hacer lo mismo con dos etiquetas, «empieza la respuesta» y «no empieza la respuesta». En cambio, `perdida_extractor` resuelve una sola clasificación para el inicio, que elige una posición entre todas las de la ventana, y otra para el fin. ¿Qué garantiza ese planteo que la clasificación de cada posición no garantiza?

*(Escribí tu respuesta acá)*

### Ejercicio 3. De los logits al texto de la respuesta

**Objetivo.** Recuperar el texto de la respuesta a partir de los logits de todas las ventanas de una pregunta.

Después del afinado, el extractor produce para cada ventana un logit de inicio y un logit de fin por posición. La respuesta predicha es el **tramo válido** del contexto con mayor puntaje. El extractor indica un tramo con dos posiciones de la ventana. La posición de inicio es la de su primera pieza. La posición de fin es la de su última pieza. El puntaje de un tramo es la suma del logit de inicio en su posición de inicio y el logit de fin en su posición de fin. Un tramo es válido si cumple tres condiciones:

- sus posiciones de inicio y de fin son posiciones del contexto;
- su posición de inicio no es posterior a su posición de fin;
- el tramo ocupa a lo sumo `LARGO_MAX_TRAMO` piezas.

`LARGO_MAX_TRAMO` vale 30. Es un tope que el extractor le impone a su predicción, para descartar tramos demasiado largos, y no el largo de las respuestas del corpus.

Para reducir la cantidad de pares de posiciones que se evalúan, los candidatos se limitan a las `N_MEJORES` posiciones con mayor logit de inicio y a las `N_MEJORES` posiciones con mayor logit de fin, que son 20 posiciones de inicio y 20 de fin. Los candidatos se eligen entre todas las posiciones de la ventana, incluidas las que no son del contexto. Las tres condiciones de validez se verifican después, sobre cada par de candidatos. Cuando el contexto de una pregunta se divide en varias ventanas, la respuesta es el tramo válido de mayor puntaje entre todas las ventanas de la pregunta.

**Enunciado:**

1. Escribí la función `mejor_tramo(l_inicio, l_fin, tramos)`. Los tres argumentos corresponden a una sola ventana:
   - `l_inicio` y `l_fin` son los arreglos de NumPy de los logits de inicio y de fin, con un logit por posición de la ventana;
   - `tramos` es un elemento de la clave `"offsets"`, la lista de los tramos de las piezas de la ventana.

   La función devuelve la tupla `(puntaje, car_a, car_b)` del tramo válido de mayor puntaje entre los candidatos. La función elige el tramo por sus posiciones y lo devuelve como el par de índices de carácter `(car_a, car_b)`. `car_a` es el índice `a` del tramo de la pieza en la posición de inicio. `car_b` es el índice `b` del tramo de la pieza en la posición de fin. Si la ventana no tiene ningún tramo válido entre los candidatos, la función devuelve `None`.

   > **Pista.** La expresión `np.argsort(x)[::-1][:N_MEJORES]` devuelve las posiciones de los `N_MEJORES` valores más altos del arreglo `x`, de mayor a menor. Esa expresión sirve para elegir los candidatos de inicio y los candidatos de fin.

2. Escribí la función `responder(logits, ventanas, ejemplos)`. Los tres argumentos corresponden a una partición:
   - `logits` es una lista con un par `(l_inicio, l_fin)` por ventana, en el orden de `ventanas`. Cada par tiene los dos arreglos que recibe `mejor_tramo`. Al terminar el entrenamiento, `afinar` calcula esta lista para las ventanas de validación y de XQuAD y se la pasa a `responder`;
   - `ventanas` es un diccionario como `ventanas_beto["val"]`;
   - `ejemplos` es la partición de la que salen las ventanas, como `val`.

   La función devuelve una lista con un texto por ejemplo, en el orden de `ejemplos`. El contexto de un ejemplo se divide en varias ventanas cuando no entra junto con la pregunta en `LARGO_MAX` posiciones. La clave `"ejemplo"` de `ventanas` indica de qué ejemplo salió cada ventana. La función aplica `mejor_tramo` a cada ventana y, para cada ejemplo, se queda con el resultado de mayor puntaje entre sus ventanas.

   Como los tramos están en índices de carácter del contexto entero, el texto del ejemplo es `contexto[car_a:car_b]`, con los índices de ese resultado. Si `mejor_tramo` devuelve `None` en todas las ventanas del ejemplo, el texto es vacío. La celda de preparación posterior a este ejercicio usa `responder`, a través de `afinar`, para obtener las respuestas de todos los modelos del laboratorio.

   > **Pista.** Para guardar el mejor resultado de cada ejemplo mientras recorrés las ventanas, podés crear un diccionario propio, que empieza vacío. La clave de ese diccionario es el índice del ejemplo, que para la ventana `k` es `ventanas["ejemplo"][k]`. El valor es la tupla `(puntaje, car_a, car_b)` de mayor puntaje que devolvió `mejor_tramo` hasta ese momento para ese ejemplo.

3. Verificá las dos funciones con los datos siguientes, que corresponden a un ejemplo inventado. El contexto tiene siete piezas, en las posiciones 4 a 10. El contexto del ejemplo se divide en dos ventanas con los mismos tramos y logits distintos.

   ```python
   ejemplos_prueba = [{"contexto": "La capital de Francia es París."}]
   tramos_prueba = [None, None, None, None, (0, 2), (3, 10), (11, 13), (14, 21),
                     (22, 24), (25, 30), (30, 31), None]
   l_inicio_1 = np.array([5.0, 0, 0, 0, 0, 0, 0, 1.0, 0, 4.0, 0, 0])
   l_fin_1 = np.array([0, 0, 6.0, 0, 0, 0, 0, 3.0, 0, 2.5, 0, 0])
   l_inicio_2 = np.array([0, 0, 0, 0, 0, 0, 0, 5.0, 0, 0, 0, 0])
   l_fin_2 = np.array([0, 0, 0, 0, 0, 0, 0, 4.0, 0, 0, 0, 0])
   ventanas_prueba = {"ejemplo": [0, 0], "offsets": [tramos_prueba, tramos_prueba]}
   logits_prueba = [(l_inicio_1, l_fin_1), (l_inicio_2, l_fin_2)]
   ```

   En la primera ventana hay dos pares con más puntaje que el tramo esperado, y ninguno de los dos es válido. El par de las posiciones 0 y 2 no es del contexto. En el par de las posiciones 9 y 7, la posición de inicio es posterior a la de fin. La segunda ventana tiene un tramo válido con más puntaje que el de la primera.

   - Imprimí el resultado de `mejor_tramo` para la primera ventana. El resultado tiene que ser `(6.5, 25, 30)`.
   - Imprimí el resultado de `responder` sobre `logits_prueba`, `ventanas_prueba` y `ejemplos_prueba`. El resultado tiene que ser `['Francia']`.

4. Verificá el límite de largo y el caso sin tramo válido con los datos siguientes. Las variables que terminan en `_largo` describen una ventana con 40 posiciones del contexto, de la 1 a la 40. La pieza de cada posición ocupa un carácter. El tramo de la posición 1 a la 40 tiene el mayor puntaje, pero ocupa 40 piezas y no es válido. El tramo de la posición 1 a la 30 ocupa 30 piezas y es válido. La única ventana de `ventanas_vacia` no tiene ninguna posición del contexto.

   ```python
   tramos_largo = [None] + [(i, i + 1) for i in range(40)] + [None]
   l_inicio_largo = np.zeros(42)
   l_inicio_largo[1] = 5.0
   l_fin_largo = np.zeros(42)
   l_fin_largo[[30, 40]] = [1.0, 5.0]
   ventanas_vacia = {"ejemplo": [0], "offsets": [[None] * 12]}
   ```

   - Imprimí el resultado de `mejor_tramo` sobre `l_inicio_largo`, `l_fin_largo` y `tramos_largo`. El resultado tiene que ser `(6.0, 0, 30)`.
   - Imprimí el resultado de `mejor_tramo` sobre `l_inicio_1`, `l_fin_1` y los tramos de `ventanas_vacia`. El resultado tiene que ser `None`.
   - Imprimí el resultado de `responder` sobre `[(l_inicio_1, l_fin_1)]`, `ventanas_vacia` y `ejemplos_prueba`. El resultado tiene que ser `['']`.

In [ ]:
# Tu código aquí

---
### Preparación. La coincidencia exacta y el F1 por palabras

El desempeño del extractor se mide comparando el texto de la respuesta predicha con las respuestas de referencia de la pregunta. Antes de la comparación, los dos textos se **normalizan**. La normalización pasa el texto a minúsculas, elimina los signos de puntuación y los artículos `el`, `la`, `los`, `las`, `un`, `una`, `unos` y `unas`, y reduce los espacios repetidos a uno. Después de la normalización, «la Casa Blanca.» y «Casa Blanca» son el mismo texto.

Con los dos textos normalizados se calculan dos cifras por pregunta:

- la **coincidencia exacta**, en inglés *exact match* o EM, que vale 1 si los dos textos son iguales y 0 si no lo son;
- el **F1 por palabras**, que es la media armónica de la precisión y el *recall* calculados sobre las palabras de los dos textos. La precisión es la proporción de palabras de la predicción que también están en la referencia. El *recall* es la proporción de palabras de la referencia que también están en la predicción.

Cuando la pregunta tiene varias respuestas de referencia, la predicción se compara con cada referencia por separado. La EM y el F1 de la pregunta son los valores más altos de esas comparaciones. Por lo tanto, a la predicción le alcanza con coincidir con una sola de las referencias. La comparación usa solo el texto de cada referencia, no el lugar que ocupa en el contexto. Si los dos textos quedan vacíos después de la normalización, por ejemplo con la referencia «la», las dos cifras valen 1. La EM y el F1 de una partición entera son el promedio de las cifras de todas sus preguntas, expresado en porcentaje.

Esta forma de medir es la del *script* de evaluación oficial de SQuAD, que los autores del corpus publicaron junto con los datos. Ese *script* elimina los artículos del inglés. La función `normalizar` de la celda elimina los del castellano.

La función `medir` calcula las dos cifras. La celda muestra la EM y el F1 de tres pares de predicción y referencia, y verifica `medir` comparando la primera respuesta de referencia de cada pregunta de `val` con todas sus referencias, lo que tiene que dar 100 en las dos cifras.

| Nombre | Qué es |
|---|---|
| `normalizar(texto)` | la función que devuelve el texto normalizado |
| `medir(ejemplos, predichas)` | la función que calcula las dos cifras. El argumento `ejemplos` es una lista de ejemplos con el formato de `val`. El argumento `predichas` es la lista que devuelve `responder`, con un texto por ejemplo en el mismo orden que `ejemplos`. La función devuelve una `Series` con la EM y el F1, en porcentaje |

In [ ]:
# ─── Preparación. La coincidencia exacta y el F1 por palabras ───────────────
ARTICULOS = re.compile(r"\b(el|la|los|las|un|una|unos|unas)\b")


def normalizar(texto):
    """Minúsculas, sin signos de puntuación, sin artículos y con un solo espacio entre palabras."""
    texto = "".join(c for c in texto.lower() if not unicodedata.category(c).startswith("P"))
    return " ".join(ARTICULOS.sub(" ", texto).split())


def em_palabras(predicha, referencia):
    return float(normalizar(predicha) == normalizar(referencia))


def f1_palabras(predicha, referencia):
    p, r = normalizar(predicha).split(), normalizar(referencia).split()
    if p == r:                                              # incluye dos textos que quedan vacíos
        return 1.0
    comunes = sum((Counter(p) & Counter(r)).values())       # palabras en común, con repetición
    if comunes == 0:
        return 0.0
    precision, recall = comunes / len(p), comunes / len(r)
    return 2 * precision * recall / (precision + recall)


def medir(ejemplos, predichas):
    """EM y F1 por palabras, en %. Cada pregunta recibe la mayor cifra entre sus referencias."""
    em = [max(em_palabras(p, r) for r in e["respuestas"]) for e, p in zip(ejemplos, predichas)]
    f1 = [max(f1_palabras(p, r) for r in e["respuestas"]) for e, p in zip(ejemplos, predichas)]
    return (100 * pd.Series({"EM": np.mean(em), "F1": np.mean(f1)})).round(1)


pares = [("la Casa Blanca.", "Casa Blanca"), ("dos poblaciones de roedores", "poblaciones de roedores"),
         ("en 1970", "2006")]
display(pd.DataFrame([{"predicha": p, "referencia": r, "EM": em_palabras(p, r),
                       "F1": round(f1_palabras(p, r), 3)} for p, r in pares]))
medir(val, [e["respuestas"][0] for e in val]).to_frame("referencias contra sí mismas")

---
### Preparación. Los dos extractores completos

La celda que sigue afina con `afinar` dos modelos sobre BETO, con la semilla 0. El primero es `BertForQuestionAnswering`, la clase de Hugging Face con la misma arquitectura que tu extractor. El segundo es tu `ExtractorBERT`. Las dos clases difieren en la inicialización de la capa lineal. `transformers` inicializa los pesos con una distribución normal de desvío 0,02 y los sesgos en cero. `nn.Linear` usa por defecto una distribución uniforme.

Al terminar, la celda construye la tabla `comparacion`, con una fila por modelo y seis columnas. Las columnas son la EM y el F1 sobre la muestra de validación de SQuAD-es, la EM y el F1 sobre XQuAD, los minutos de afinado y los parámetros entrenables en miles. El Ejercicio 4 le agrega una fila. La celda muestra además cinco preguntas de XQuAD, elegidas al azar con la semilla 0, con su respuesta de referencia y la respuesta de tu extractor.

> **Nota. La verificación de tu código.** La comparación entre los dos modelos sirve para verificar tu código de los ejercicios 1 a 3. Los dos modelos usan tu función `ubicar`, que da la posición de la respuesta en cada ventana de entrenamiento. Los dos modelos usan también tus funciones `mejor_tramo` y `responder`, que convierten los logits en textos. Solo tu extractor usa `ExtractorBERT` y `perdida_extractor`. Un error en ese código puede no interrumpir la ejecución, porque el afinado corre igual con posiciones equivocadas. Con una sola semilla, la inicialización distinta y el azar del afinado pueden separar el F1 de los dos modelos en uno o dos puntos. Si tu extractor queda más de 5 puntos de F1 por debajo de `BertForQuestionAnswering`, el error está en `ExtractorBERT` o en `perdida_extractor`. Si los dos modelos quedan por debajo de 60 de F1 en validación, el error está en el código que comparten.

| Nombre | Qué es |
|---|---|
| `construir_hf` | la función sin argumentos que devuelve un `BertForQuestionAnswering` sobre BETO, sin afinar |
| `construir_mano` | la función sin argumentos que devuelve un `ExtractorBERT`, sin afinar |
| `resultados` | un diccionario con las claves `"hf_s0"` y `"mano_s0"`. El valor de cada clave es el diccionario que devuelve `afinar` para ese modelo |
| `resumen(r)` | la función que recibe un diccionario `r` de los que devuelve `afinar` y devuelve una `Series` con los seis valores de su fila de `comparacion` |
| `comparacion` | un `DataFrame` cuyo índice tiene el nombre de cada modelo |

In [ ]:
# ─── Preparación. Los dos extractores completos, semilla 0 ──────────────────
def construir_hf():
    return BertForQuestionAnswering.from_pretrained(BETO)


def construir_mano():
    return ExtractorBERT()


resultados = {"hf_s0": afinar("qa_hf_s0", construir_hf, perdida_hf, ventanas_beto, 0),
              "mano_s0": afinar("qa_mano_s0", construir_mano, perdida_extractor, ventanas_beto, 0)}


def resumen(r):
    """La fila de la tabla comparativa para un resultado de afinar."""
    en_val, en_xquad = medir(val, r["val"]), medir(xquad, r["xquad"])
    return pd.Series({"EM val.": en_val["EM"], "F1 val.": en_val["F1"],
                      "EM XQuAD": en_xquad["EM"], "F1 XQuAD": en_xquad["F1"],
                      "minutos": r["minutos"],
                      "parámetros entrenables (miles)": round(r["entrenables"] / 1e3, 1)})


NOMBRES = {"hf_s0": "BertForQuestionAnswering, semilla 0", "mano_s0": "a mano, semilla 0"}
comparacion = pd.DataFrame({NOMBRES[k]: resumen(r) for k, r in resultados.items()}).T
display(comparacion)

muestra = np.random.default_rng(0).choice(len(xquad), 5, replace=False)
pd.DataFrame({"pregunta": [xquad[i]["pregunta"] for i in muestra],
              "referencia": [xquad[i]["respuestas"][0] for i in muestra],
              "tu extractor": [resultados["mano_s0"]["xquad"][i] for i in muestra]})

### Ejercicio 4. El encoder congelado

**Objetivo.** Afinar solo la cabeza del extractor, con el encoder congelado, y comparar el resultado con el del extractor afinado entero.

La sección 3.2 de la Clase 2 congeló el encoder del clasificador de reseñas y entrenó solo la cabeza, con un `lr` de `1e-3`. La accuracy de prueba del clasificador fue de 60,7 % afinado entero y de 51,0 % con el encoder congelado. Este ejercicio repite esa comparación con el extractor. Respecto de `mano_s0`, cambian los parámetros que se entrenan y el `lr`, como en la sección 3.2.

**Enunciado:**

1. Escribí la función `construir_congelado()`. La función no recibe argumentos, crea un `ExtractorBERT` sobre BETO, desactiva el gradiente de todos los parámetros de su atributo `encoder` y devuelve el modelo.
2. Afiná el extractor congelado con `afinar`, con el nombre `"qa_congelado_s0"`, la función `construir_congelado`, la pérdida `perdida_extractor`, las ventanas `ventanas_beto`, la semilla 0 y un `lr` de `1e-3`. Guardá el resultado en una variable llamada `congelado_s0`. Aplicale `resumen` a `congelado_s0` y agregá la `Series` que devuelve como una fila nueva de `comparacion`, con la etiqueta `"a mano, encoder congelado, semilla 0"`. Mostrá `comparacion`.
3. Medí con qué frecuencia dos preguntas distintas sobre un mismo contexto reciben el mismo texto como respuesta. La medición usa los pares de preguntas de XQuAD que comparten contexto. Dos preguntas comparten contexto cuando la clave `"contexto"` de sus ejemplos tiene el mismo texto. Para encontrarlas, agrupá los índices de los ejemplos de `xquad` en un diccionario que tenga el texto del contexto como clave y la lista de índices como valor. Cada par está formado por dos índices de una misma lista, y cada par se cuenta una sola vez. La medición se repite sobre tres fuentes de respuestas:
   - la primera respuesta de referencia de cada pregunta;
   - las respuestas de `resultados["mano_s0"]["xquad"]`;
   - las respuestas de `congelado_s0["xquad"]`.

   Para cada fuente, calculá el porcentaje de pares cuyas dos respuestas son iguales carácter por carácter, sin normalizar. Guardá los tres porcentajes en una `Series` llamada `mismo_texto`, con una etiqueta por fuente. Imprimí la cantidad de pares y mostrá `mismo_texto` con un decimal.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Con el encoder congelado, el extractor pierde una parte de su F1 mucho mayor que la parte de la accuracy que perdió el clasificador de reseñas en la sección 3.2 de la Clase 2. Explicá esa diferencia a partir de la información que tiene que contener el vector de salida de cada posición del contexto para que una misma capa lineal, aplicada a todas las preguntas, le asigne el mayor logit de inicio a la primera pieza de la respuesta. Usá `mismo_texto` para sostener la explicación.

*(Escribí tu respuesta acá)*

### Ejercicio 5. Las respuestas de referencia

**Objetivo.** Medir tres propiedades de las respuestas de referencia de las dos particiones de evaluación, para decidir con cuál se reporta el desempeño del extractor.

El laboratorio evalúa los extractores sobre dos particiones, la muestra de validación de SQuAD-es y XQuAD. Las dos tienen preguntas de la validación del SQuAD original, traducidas de maneras distintas. La función `medir` compara cada predicción con las respuestas de referencia de su pregunta. Este ejercicio mide tres propiedades de esas respuestas en las dos particiones.

**Enunciado:**

1. Construí un `DataFrame` llamado `referencias`, con una columna para `val`, otra para `xquad` y una fila para cada una de estas tres propiedades:
   - la cantidad media de respuestas de referencia por pregunta;
   - el porcentaje de respuestas de referencia cuya última palabra pertenece al conjunto `FUNCION`. Para obtener la última palabra de una respuesta, separá la respuesta por los espacios, tomá la última parte, pasala a minúsculas y quitale los signos `.,;:"` de los extremos;
   - el porcentaje de preguntas que tienen al menos dos respuestas de referencia distintas después de aplicarles `normalizar`.

   El conjunto `FUNCION` reúne palabras de función, que son las que cumplen un papel gramatical sin aportar contenido propio, como los artículos, las preposiciones, las conjunciones, los pronombres y los posesivos. Copiá el conjunto en tu celda.

   ```python
   FUNCION = {"de", "del", "la", "el", "los", "las", "en", "y", "que", "un", "una",
              "para", "con", "a", "por", "al", "se", "su"}
   ```

   Mostrá la tabla con un decimal.
2. Imprimí cinco respuestas de referencia de `val`, distintas entre sí, cuya última palabra pertenece a `FUNCION`, cada una junto con la pregunta a la que responde.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Decí con cuál de las dos particiones reportarías el desempeño del extractor, y justificalo con la tabla `referencias` y con las ventanas del Ejercicio 1 cuyo texto recuperado no coincide con la respuesta de referencia.

*(Escribí tu respuesta acá)*

---
## Bloque opcional. La época completa

Este bloque no se corrige. Ninguna celda posterior usa lo que el bloque define. La celda afina tu `ExtractorBERT` durante una época sobre los 87.595 ejemplos de entrenamiento de SQuAD-es, en lugar de los 10.000 de los ejercicios, y muestra su fila de la tabla junto a la de `mano_s0`. La época completa tiene unos 5.500 pasos, casi nueve veces los de `mano_s0`. Con una T4, el afinado se estima del orden de quince minutos.

En la corrida de la cátedra, la época completa llevó el F1 de validación de alrededor de 69 a alrededor de 77, y el de XQuAD de 67 a 78, con el mismo código. Esa cifra está a la altura de un BETO afinado para preguntas y respuestas en castellano que está publicado en Hugging Face, `mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es`, que sobre la validación de SQuAD-es da 76,8. Lo que separa al extractor de los ejercicios de ese modelo publicado es la cantidad de datos de entrenamiento, no la arquitectura.

La celda viene desactivada. Para ejecutarla, cambiá el valor de `EPOCA_COMPLETA` a `True`. Ese cambio es la única modificación permitida fuera de las celdas de actividad.

In [ ]:
# ─── Bloque opcional. Una época sobre los 87.595 ejemplos ───────────────────
EPOCA_COMPLETA = False                              # True para ejecutar el bloque

if EPOCA_COMPLETA:
    ventanas_completo = {"ent": armar_ventanas(ent_completo, tok_beto, ubicar),
                         "val": ventanas_beto["val"], "xquad": ventanas_beto["xquad"]}
    completo = afinar("qa_mano_completo", construir_mano, perdida_extractor, ventanas_completo, 0)
    display(pd.DataFrame({"a mano, 10.000 ejemplos": resumen(resultados["mano_s0"]),
                          "a mano, 87.595 ejemplos": resumen(completo)}).T)
else:
    print("El bloque opcional no se ejecutó, porque EPOCA_COMPLETA vale False.")

---
## Antes de entregar

Revisá esta lista:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] En la verificación del Ejercicio 1, más del 90 % de las ventanas con la respuesta entera recuperan exactamente la respuesta de referencia.
- [ ] La pérdida de `demo` sin afinar, en el Ejercicio 2, está a menos de medio punto de $\ln L$.
- [ ] Las verificaciones del Ejercicio 3 dan `(6.5, 25, 30)`, `['Francia']`, `(6.0, 0, 30)`, `None` y `['']`.
- [ ] `comparacion` tiene tres filas. La fila del encoder congelado se afinó con un `lr` de `1e-3`.
- [ ] Respondí las preguntas de análisis de los ejercicios 1, 2, 4 y 5.
- [ ] No modifiqué ninguna celda fuera de las de actividad, salvo el interruptor del bloque opcional.

---
## Cierre

Este laboratorio construyó la cabeza de SQuAD que la sección 1.3 de la Clase 2 mostraba en la figura del artículo de BERT. Como en la Parte A, el modelo ocupa pocas líneas. El resto del trabajo estuvo en el paso de caracteres a posiciones, en la división de los contextos en ventanas y en la recuperación del texto a partir de los logits.

Dos prácticas del laboratorio se aplican a cualquier experimento con modelos preentrenados:

- verificar la conversión de los datos al formato del modelo recuperando los datos originales, antes de entrenar;
- revisar las respuestas de referencia de un corpus antes de reportar una cifra medida contra ellas.

La Unidad 4 deja el encoder y estudia GPT, un modelo formado solo por bloques de decoder.